In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

df = pd.read_csv('../data/CUAD_v1/master_clauses.csv')
print(f"Shape: {df.shape}")

c:\Users\thein\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Shape: (510, 83)


In [2]:
target_clauses = [
    'Non-Compete',
    'Termination For Convenience',
    'Change Of Control',
    'Anti-Assignment',
    'License Grant',
    'Audit Rights',
    'Cap On Liability',
    'Governing Law'
]

records = []
for _, row in df.iterrows():
    for clause in target_clauses:
        text = row[clause]
        if pd.notna(text) and str(text).strip() not in ['', 'nan']:
            records.append({
                'clause_type': clause,
                'text': str(text).strip()
            })

flat_df = pd.DataFrame(records)
print(f"Total samples: {len(flat_df)}")
print(flat_df['clause_type'].value_counts())

Total samples: 4080
clause_type
Non-Compete                    510
Termination For Convenience    510
Change Of Control              510
Anti-Assignment                510
License Grant                  510
Audit Rights                   510
Cap On Liability               510
Governing Law                  510
Name: count, dtype: int64


In [3]:
# Single model name used everywhere
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

le = LabelEncoder()
flat_df['label'] = le.fit_transform(flat_df['clause_type'])
print(f"Classes: {list(le.classes_)}")

train_df, test_df = train_test_split(
    flat_df, test_size=0.2, random_state=42, stratify=flat_df['label']
)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

class ClauseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

train_dataset = ClauseDataset(train_df['text'], train_df['label'], tokenizer)
test_dataset = ClauseDataset(test_df['text'], test_df['label'], tokenizer)
print(f"Datasets ready")

Classes: ['Anti-Assignment', 'Audit Rights', 'Cap On Liability', 'Change Of Control', 'Governing Law', 'License Grant', 'Non-Compete', 'Termination For Convenience']
Train: 3264, Test: 816
Datasets ready


In [6]:
from torch.nn import CrossEntropyLoss
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from transformers import Trainer, TrainingArguments

# Redefine compute_metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

# Compute class weights from training labels
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label'].values),
    y=train_df['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights: {dict(zip(le.classes_, class_weights.round(3)))}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = CrossEntropyLoss(weight=class_weights_tensor)(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_),
    ignore_mismatched_sizes=True
)
model = model.to(device)

training_args = TrainingArguments(
    output_dir='../models/bert-clauses-weighted',
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available()
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting weighted training...")
trainer.train()

Class weights: {'Anti-Assignment': np.float64(1.0), 'Audit Rights': np.float64(1.0), 'Cap On Liability': np.float64(1.0), 'Change Of Control': np.float64(1.0), 'Governing Law': np.float64(1.0), 'License Grant': np.float64(1.0), 'Non-Compete': np.float64(1.0), 'Termination For Convenience': np.float64(1.0)}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5432.79it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

Starting weighted training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.488163,1.326674,0.537990,0.555687
2,1.125219,1.110347,0.564951,0.597749
3,1.060719,1.109376,0.566176,0.610523
4,1.049710,1.098553,0.568627,0.606224
5,1.046901,1.100286,0.568627,0.603830
6,1.055463,1.098688,0.567402,0.602718


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=1224, training_loss=1.1971464297350716, metrics={'train_runtime': 4907.002, 'train_samples_per_second': 3.991, 'train_steps_per_second': 0.249, 'total_flos': 2576522247929856.0, 'train_loss': 1.1971464297350716, 'epoch': 6.0})

In [7]:
model.eval()
all_preds, all_labels = [], []
test_loader = DataLoader(test_dataset, batch_size=16)

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_))

                             precision    recall  f1-score   support

            Anti-Assignment       0.89      0.69      0.77       102
               Audit Rights       1.00      0.47      0.64       102
           Cap On Liability       1.00      0.58      0.73       102
          Change Of Control       0.21      0.91      0.35       102
              Governing Law       1.00      0.86      0.93       102
              License Grant       1.00      0.53      0.69       102
                Non-Compete       1.00      0.18      0.30       102
Termination For Convenience       0.97      0.31      0.47       102

                   accuracy                           0.57       816
                  macro avg       0.88      0.57      0.61       816
               weighted avg       0.88      0.57      0.61       816



In [8]:
import numpy as np
from sklearn.metrics import f1_score
import torch
from torch.utils.data import DataLoader

# Get raw probabilities
model.eval()
all_probs, all_labels = [], []
test_loader = DataLoader(test_dataset, batch_size=16)

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Find optimal threshold per class
print("Optimal thresholds per class:")
best_thresholds = {}
for i, class_name in enumerate(le.classes_):
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = (all_probs[:, i] >= thresh).astype(int)
        labels_binary = (all_labels == i).astype(int)
        f1 = f1_score(labels_binary, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    best_thresholds[class_name] = round(best_thresh, 2)
    print(f"  {class_name}: threshold={best_thresh:.2f}, f1={best_f1:.3f}")

Optimal thresholds per class:
  Anti-Assignment: threshold=0.50, f1=0.778
  Audit Rights: threshold=0.20, f1=0.640
  Cap On Liability: threshold=0.15, f1=0.728
  Change Of Control: threshold=0.10, f1=0.346
  Governing Law: threshold=0.10, f1=0.926
  License Grant: threshold=0.15, f1=0.692
  Non-Compete: threshold=0.10, f1=0.371
  Termination For Convenience: threshold=0.25, f1=0.474


In [9]:
final_preds = np.zeros(len(all_labels), dtype=int)

for i in range(len(all_labels)):
    best_class, best_prob = 0, 0
    for class_idx, class_name in enumerate(le.classes_):
        thresh = best_thresholds[class_name]
        prob = all_probs[i, class_idx]
        if prob >= thresh and prob > best_prob:
            best_prob = prob
            best_class = class_idx
    final_preds[i] = best_class

print("Final Classification Report with Tuned Thresholds:")
print(classification_report(all_labels, final_preds, target_names=le.classes_))

# Save thresholds
import pickle
with open('../models/class_thresholds.pkl', 'wb') as f:
    pickle.dump(best_thresholds, f)
print("Thresholds saved")

Final Classification Report with Tuned Thresholds:
                             precision    recall  f1-score   support

            Anti-Assignment       0.90      0.69      0.78       102
               Audit Rights       1.00      0.47      0.64       102
           Cap On Liability       1.00      0.58      0.73       102
          Change Of Control       0.21      0.92      0.35       102
              Governing Law       1.00      0.86      0.93       102
              License Grant       1.00      0.53      0.69       102
                Non-Compete       1.00      0.18      0.30       102
Termination For Convenience       0.97      0.31      0.47       102

                   accuracy                           0.57       816
                  macro avg       0.89      0.57      0.61       816
               weighted avg       0.89      0.57      0.61       816

Thresholds saved


In [10]:
import pickle, os
os.makedirs('../models/bert-clauses', exist_ok=True)

model.save_pretrained('../models/bert-clauses')
tokenizer.save_pretrained('../models/bert-clauses')

with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Saved successfully")
print(f"Classes: {list(le.classes_)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

Saved successfully
Classes: ['Anti-Assignment', 'Audit Rights', 'Cap On Liability', 'Change Of Control', 'Governing Law', 'License Grant', 'Non-Compete', 'Termination For Convenience']
